# 02. latent predictor와 EMA target encoder

목표: context encoder와 predictor는 gradient로, target encoder는 EMA로만 갱신하는 책임 분리를 scalar toy model로 구현한다. 이미지·ViT를 재현하지 않으며 학습 loop의 원리만 보존한다.

In [ ]:
import random

rng = random.Random(11)
samples = []
for _ in range(256):
    shared = rng.uniform(-1.0, 1.0)
    context_view = shared + rng.gauss(0.0, 0.05)
    target_view = 2.0 * shared + rng.gauss(0.0, 0.05)
    samples.append((context_view, target_view))

samples[:3]

In [ ]:
online_weight = 1.0       # context encoder θ
predictor_weight = 0.2    # predictor φ
target_weight = online_weight  # target encoder θ̄는 online 복사본으로 시작
learning_rate = 0.05
momentum = 0.99
loss_history = []

for epoch in range(80):
    total_loss = 0.0
    for context_view, target_view in samples:
        context_latent = online_weight * context_view
        prediction = predictor_weight * context_latent
        # target는 stop-gradient: 아래 값으로 gradient를 계산하지 않는다.
        target_latent = target_weight * target_view
        error = prediction - target_latent
        total_loss += 0.5 * error * error

        grad_predictor = error * context_latent
        grad_online = error * predictor_weight * context_view
        predictor_weight -= learning_rate * grad_predictor
        online_weight -= learning_rate * grad_online

        # optimizer step 뒤 target encoder를 EMA로만 갱신한다.
        target_weight = (
            momentum * target_weight
            + (1.0 - momentum) * online_weight
        )
    loss_history.append(total_loss / len(samples))

print(f"첫 epoch loss: {loss_history[0]:.6f}")
print(f"마지막 epoch loss: {loss_history[-1]:.6f}")
print(f"online={online_weight:.4f}, predictor={predictor_weight:.4f}, target={target_weight:.4f}")

In [ ]:
# loss가 전반적으로 줄었는지 구간 평균으로 확인한다.
first_window = sum(loss_history[:10]) / 10
last_window = sum(loss_history[-10:]) / 10
print("초기 10 epoch 평균:", first_window)
print("마지막 10 epoch 평균:", last_window)
assert last_window < first_window

## 실험 과제

- momentum을 0.9, 0.99, 0.999로 바꿔 안정성과 지연을 비교한다.
- target_weight에도 gradient update를 적용해 보고 왜 논문의 target branch와 다른지 설명한다.
- context와 target의 공유 정보가 없는 random data로 바꾸고 loss가 주는 신호를 관찰한다.
- 실제 구현에서는 scalar가 `[batch, target_patch, embedding_dim]` tensor로 확장됨을 스케치한다.